In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)

VOL = "/Volumes/workspace/default/maplebank"

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])

df_txn      = spark.read.option("header", True).schema(txn_schema).csv(f"{VOL}/fact_transactions.csv")
df_customer = spark.read.option("header", True).csv(f"{VOL}/dim_customer.csv")
df_branch   = spark.read.option("header", True).csv(f"{VOL}/dim_branch.csv")
print(f"Loaded {df_txn.count():,} transactions")


Loaded 10,000 transactions


In [0]:
# Real accounts have debits and credits. Simulate: deposits are +, everything else is -
df_signed = df_txn.withColumn(
    "signed_amount",
    F.when(F.col("transaction_type") == "PAYROLL_DEPOSIT", F.col("amount_cad"))
     .otherwise(-F.col("amount_cad"))
)

w_running = Window.partitionBy("account_id") \
                  .orderBy("transaction_timestamp") \
                  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_balance = df_signed.withColumn(
    "running_balance",
    F.round(F.sum("signed_amount").over(w_running), 2)
)

# Watch one account's story unfold:
df_balance.filter(F.col("account_id") == df_balance.select("account_id").first()[0]) \
    .select("transaction_timestamp", "transaction_type", "signed_amount", "running_balance") \
    .orderBy("transaction_timestamp") \
    .show(10, truncate=False)

+---------------------+----------------+-------------+---------------+
|transaction_timestamp|transaction_type|signed_amount|running_balance|
+---------------------+----------------+-------------+---------------+
|2025-11-02 12:25:00  |ATM_WITHDRAWAL  |-3023.57     |-3023.57       |
|2025-11-04 17:13:00  |BILL_PAYMENT    |-3038.65     |-6062.22       |
|2025-11-05 07:31:00  |PAYROLL_DEPOSIT |935.26       |-5126.96       |
|2025-11-23 06:42:00  |INTERAC_DEBIT   |-716.02      |-5842.98       |
|2025-11-24 17:13:00  |BILL_PAYMENT    |-3625.93     |-9468.91       |
|2025-11-30 09:19:00  |BILL_PAYMENT    |-1264.06     |-10732.97      |
+---------------------+----------------+-------------+---------------+



In [0]:
days = lambda n: n * 86400   # rangeBetween works in seconds on a long timestamp

w_30d = Window.partitionBy("customer_id") \
              .orderBy(F.col("transaction_timestamp").cast("long")) \
              .rangeBetween(-days(30), 0)

df_rolling = df_txn.withColumn(
    "spend_30d",
    F.round(F.sum("amount_cad").over(w_30d), 2)
).withColumn(
    "txn_count_30d",
    F.count("*").over(w_30d)
)

df_rolling.select("customer_id", "transaction_timestamp", "amount_cad",
                  "spend_30d", "txn_count_30d") \
    .orderBy("customer_id", "transaction_timestamp") \
    .show(8, truncate=False)
    

+-----------+---------------------+----------+---------+-------------+
|customer_id|transaction_timestamp|amount_cad|spend_30d|txn_count_30d|
+-----------+---------------------+----------+---------+-------------+
|CUST100001 |2025-11-01 06:57:00  |2093.58   |2093.58  |1            |
|CUST100001 |2025-11-01 07:59:00  |1077.25   |3170.83  |2            |
|CUST100001 |2025-11-01 19:39:00  |4034.35   |7205.18  |3            |
|CUST100001 |2025-11-06 09:40:00  |1693.08   |8898.26  |4            |
|CUST100001 |2025-11-07 04:42:00  |788.47    |9686.73  |5            |
|CUST100001 |2025-11-08 02:46:00  |3726.71   |13413.44 |6            |
|CUST100001 |2025-11-10 18:37:00  |1593.37   |15006.81 |7            |
|CUST100001 |2025-11-15 20:28:00  |2086.2    |17093.01 |8            |
+-----------+---------------------+----------+---------+-------------+
only showing top 8 rows


In [0]:
df_cust_branch = df_txn.groupBy("branch_id", "customer_id") \
    .agg(F.round(F.sum("amount_cad"), 2).alias("total_spend"))

w_rank = Window.partitionBy("branch_id").orderBy(F.col("total_spend").desc())

df_top3 = df_cust_branch \
    .withColumn("rank_in_branch", F.row_number().over(w_rank)) \
    .filter(F.col("rank_in_branch") <= 3)

df_top3.orderBy("branch_id", "rank_in_branch").show(9, truncate=False)

+----------+-----------+-----------+--------------+
|branch_id |customer_id|total_spend|rank_in_branch|
+----------+-----------+-----------+--------------+
|BR_CAL_012|CUST100049 |99214.66   |1             |
|BR_CAL_012|CUST100292 |77432.13   |2             |
|BR_CAL_012|CUST100193 |71053.19   |3             |
|BR_EDM_013|CUST100434 |88735.79   |1             |
|BR_EDM_013|CUST100300 |72379.39   |2             |
|BR_EDM_013|CUST100880 |71502.15   |3             |
|BR_HAL_015|CUST100531 |71868.02   |1             |
|BR_HAL_015|CUST100728 |59157.21   |2             |
|BR_HAL_015|CUST100591 |58706.69   |3             |
+----------+-----------+-----------+--------------+
only showing top 9 rows


In [0]:
# Simulate what source systems actually do: resend a transaction (duplicate ID, newer timestamp)
dupe = df_txn.limit(1) \
    .withColumn("transaction_timestamp", F.col("transaction_timestamp") + F.expr("INTERVAL 2 HOURS")) \
    .withColumn("amount_cad", F.lit(9999.99))
df_with_dupes = df_txn.union(dupe)
print(f"With duplicate injected: {df_with_dupes.count():,}")

# Keep only the LATEST version per transaction_id
w_dedup = Window.partitionBy("transaction_id").orderBy(F.col("transaction_timestamp").desc())

df_deduped = df_with_dupes \
    .withColumn("rn", F.row_number().over(w_dedup)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

print(f"After dedup-keep-latest:  {df_deduped.count():,}")

With duplicate injected: 10,001
After dedup-keep-latest:  10,000


In [0]:
# Approach A — NATIVE functions (Catalyst-optimized, fast):
df_native = df_customer.withColumn(
    "sin_masked",
    F.concat(F.lit("***-***-"), F.substring(F.col("sin"), 8, 3))
)

# Approach B — Python UDF (works, but a black box to the optimizer):
from pyspark.sql.types import StringType as ST
@F.udf(returnType=ST())
def mask_sin_udf(sin):
    return f"***-***-{sin[-3:]}" if sin else "***-***-***"

df_udf = df_customer.withColumn("sin_masked", mask_sin_udf(F.col("sin")))

df_native.select("customer_id", "sin", "sin_masked").show(3)
print("Both produce the same result — but native wins on performance.")
print("Interview line: 'UDFs are opaque to Catalyst; I default to native functions.'")

+-----------+-----------+-----------+
|customer_id|        sin| sin_masked|
+-----------+-----------+-----------+
| CUST100001|338-617-716|***-***--71|
| CUST100002|303-833-765|***-***--76|
| CUST100003|325-559-703|***-***--70|
+-----------+-----------+-----------+
only showing top 3 rows
Both produce the same result — but native wins on performance.
Interview line: 'UDFs are opaque to Catalyst; I default to native functions.'


In [0]:
df_joined = df_txn.join(F.broadcast(df_branch), on="branch_id", how="left")

df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [branch_id#13773, transaction_id#13770, customer_id#13771, account_id#13772, transaction_date#13774, transaction_timestamp#13775, amount_cad#13776, transaction_type#13777, merchant_name#13778, channel#13779, branch_name#13798, city#13799, province#13800, transit_number#13801]
         +- PhotonBroadcastHashJoin [branch_id#13773], [branch_id#13797], LeftOuter, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- FileScan csv [transaction_id#13770,customer_id#13771,account_id#13772,branch_id#13773,transaction_date#13774,transaction_timestamp#13775,amount_cad#13776,transaction_type#13777,merchant_name#13778,channel#13779] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/maplebank/fact_transactions.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: